# Cyclic Voltammetry Analysis
**Experiment date:** 2026-03-31  
**Electrolyte:** 1 M NaCl  
**Conditions:**
- `pyrroleDBS` — bare polypyrrole/DBS film
- `1cycle_pyrroleDBS_decellchick` — 1-cycle PPy/DBS on decellularized chicken scaffold
- `0cycle_nopyrroleDBS_decellchick` — decellularized chicken scaffold, no PPy (control)

**Scan rates:** 10, 50, 100, 200 mV/s

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from scipy.signal import find_peaks, savgol_filter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 9,
})

## 1. Load and Parse All CSV Files

In [ ]:
DATA_DIR = Path('cyclic_voltametry_exps')

def parse_cv_file(filepath):
    """Parse a CV CSV file (UTF-16, multi-scan format).
    Returns a list of DataFrames, one per scan, with columns ['V', 'uA'].
    """
    with open(filepath, 'rb') as f:
        raw = f.read()
    # Detect UTF-16 BOM and decode
    if raw[:2] in (b'\xff\xfe', b'\xfe\xff'):
        text = raw.decode('utf-16')
    else:
        text = raw.decode('utf-8', errors='replace')

    lines = [l.strip() for l in text.splitlines()]

    # Find the units row (contains 'V' and 'A')
    header_idx = None
    for i, line in enumerate(lines):
        parts = [p.strip() for p in line.split(',')]
        non_empty = [p for p in parts if p]
        if non_empty and all(p in ('V', 'µA', 'mA', 'A', 'uA') for p in non_empty):
            header_idx = i
            ncols = len(non_empty)
            break

    if header_idx is None:
        raise ValueError(f"Could not find units row in {filepath}")

    n_scans = ncols // 2

    # Read data rows
    data_rows = []
    for line in lines[header_idx + 1:]:
        parts = [p.strip() for p in line.split(',')]
        nums = []
        for p in parts:
            if p == '':
                continue
            try:
                nums.append(float(p))
            except ValueError:
                pass
        if len(nums) >= ncols:
            data_rows.append(nums[:ncols])

    if not data_rows:
        return []

    arr = np.array(data_rows)
    scans = []
    for s in range(n_scans):
        df = pd.DataFrame({
            'V': arr[:, s * 2],
            'uA': arr[:, s * 2 + 1]
        })
        scans.append(df)
    return scans


# Metadata parsed from filenames
conditions = {
    'pyrroleDBS': {
        'label': 'PPy/DBS (bare film)',
        'color_base': 'Blues',
    },
    '1cycle_pyrroleDBS_decellchick': {
        'label': 'PPy/DBS + decell chicken (1 cycle)',
        'color_base': 'Greens',
    },
    '0cycle_nopyrroleDBS_decellchick': {
        'label': 'Decell chicken only (control)',
        'color_base': 'Reds',
    },
}

scan_rates = [10, 50, 100, 200]  # mV/s

# Map condition key -> scan rate -> list of scan DataFrames
all_data = {}

for f in sorted(DATA_DIR.glob('*.csv')):
    name = f.stem
    # Determine condition and scan rate
    matched_cond = None
    for ckey in conditions:
        if ckey in name:
            matched_cond = ckey
            break
    if matched_cond is None:
        print(f'  [skip] {f.name} — condition not recognised')
        continue

    rate = None
    for r in scan_rates:
        if f'{r}mVperSec' in name:
            rate = r
            break
    if rate is None:
        print(f'  [skip] {f.name} — scan rate not recognised')
        continue

    scans = parse_cv_file(f)
    all_data.setdefault(matched_cond, {})[rate] = scans
    print(f'  Loaded {f.name}: {len(scans)} scan(s), {len(scans[0]) if scans else 0} points each')

print('\nDone loading.')

## 2. Plot CV Curves — All Conditions and Scan Rates

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=False)

for ax, (ckey, cinfo) in zip(axes, conditions.items()):
    cmap = cm.get_cmap(cinfo['color_base'])
    rates_available = sorted(all_data.get(ckey, {}).keys())
    colors = [cmap(0.4 + 0.55 * i / max(len(rates_available) - 1, 1))
              for i in range(len(rates_available))]

    for color, rate in zip(colors, rates_available):
        scans = all_data[ckey][rate]
        # Plot only the last (most stable) scan
        df = scans[-1]
        ax.plot(df['V'], df['uA'], color=color, lw=1.5, label=f'{rate} mV/s')

    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
    ax.set_xlabel('Potential (V)')
    ax.set_ylabel('Current (µA)')
    ax.set_title(cinfo['label'])
    ax.legend(title='Scan rate', loc='best')
    ax.grid(True, alpha=0.25)

plt.suptitle('Cyclic Voltammetry — 1 M NaCl (last scan per condition)', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('cv_all_conditions.png', bbox_inches='tight')
plt.show()

## 3. Compare Conditions at Each Scan Rate

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

condition_colors = {'pyrroleDBS': '#1f77b4',
                    '1cycle_pyrroleDBS_decellchick': '#2ca02c',
                    '0cycle_nopyrroleDBS_decellchick': '#d62728'}

for ax, rate in zip(axes, scan_rates):
    for ckey, cinfo in conditions.items():
        if rate not in all_data.get(ckey, {}):
            continue
        scans = all_data[ckey][rate]
        df = scans[-1]
        ax.plot(df['V'], df['uA'], color=condition_colors[ckey],
                lw=1.8, label=cinfo['label'])

    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
    ax.set_xlabel('Potential (V)')
    ax.set_ylabel('Current (µA)')
    ax.set_title(f'{rate} mV/s')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.25)

plt.suptitle('CV Comparison by Scan Rate — 1 M NaCl', fontsize=14)
plt.tight_layout()
plt.savefig('cv_by_scan_rate.png', bbox_inches='tight')
plt.show()

## 4. Redox Peak Detection

Peak detection strategy:
- Smooth with Savitzky–Golay filter to reduce noise
- Split scan into **forward** (oxidation, dV/dt > 0) and **reverse** (reduction, dV/dt < 0) sweeps
- Find local maxima on forward sweep → oxidation peaks
- Find local minima on reverse sweep → reduction peaks
- Redox midpoint potential: E½ = (E_ox + E_red) / 2

In [ ]:
def find_redox_peaks(df, smooth_window=15, smooth_poly=3,
                     peak_prominence=10, peak_height_frac=0.05):
    """
    Returns dict with oxidation and reduction peak info.
    Splits the CV trace into forward (anodic) and reverse (cathodic) sweeps.
    """
    V = df['V'].values
    I = df['uA'].values

    # Smooth
    wl = min(smooth_window, len(I) - 1 if len(I) % 2 == 0 else len(I))
    if wl % 2 == 0:
        wl -= 1
    wl = max(wl, 5)
    I_smooth = savgol_filter(I, window_length=wl, polyorder=smooth_poly)

    # Detect direction changes
    dV = np.diff(V)
    # Forward sweep: increasing V
    fwd_mask = np.concatenate(([False], dV > 0))
    rev_mask = np.concatenate(([False], dV < 0))

    I_range = I_smooth.max() - I_smooth.min()
    min_height = peak_height_frac * I_range

    results = {'oxidation': [], 'reduction': []}

    # Oxidation peaks (maxima in forward sweep)
    if fwd_mask.sum() > 5:
        I_fwd = np.where(fwd_mask, I_smooth, np.nan)
        V_fwd = V.copy()
        idx_fwd = np.where(fwd_mask)[0]
        I_fwd_vals = I_smooth[idx_fwd]
        peaks, props = find_peaks(I_fwd_vals,
                                  prominence=max(peak_prominence, min_height),
                                  distance=5)
        for p in peaks:
            results['oxidation'].append({
                'V': V_fwd[idx_fwd[p]],
                'I': I_smooth[idx_fwd[p]],
                'prominence': props['prominences'][list(peaks).index(p)]
            })

    # Reduction peaks (minima in reverse sweep = maxima of negated signal)
    if rev_mask.sum() > 5:
        idx_rev = np.where(rev_mask)[0]
        I_rev_vals = I_smooth[idx_rev]
        peaks_r, props_r = find_peaks(-I_rev_vals,
                                      prominence=max(peak_prominence, min_height),
                                      distance=5)
        for p in peaks_r:
            results['reduction'].append({
                'V': V[idx_rev[p]],
                'I': I_smooth[idx_rev[p]],
                'prominence': props_r['prominences'][list(peaks_r).index(p)]
            })

    return results, I_smooth


print('Peak detector ready.')

In [ ]:
# Run peak detection on the last scan of each condition × scan rate
peak_results = {}  # ckey -> rate -> {'oxidation': [...], 'reduction': [...]}

for ckey in conditions:
    peak_results[ckey] = {}
    for rate in scan_rates:
        if rate not in all_data.get(ckey, {}):
            continue
        scans = all_data[ckey][rate]
        df = scans[-1]
        peaks, _ = find_redox_peaks(df)
        peak_results[ckey][rate] = peaks

        ox_str = ', '.join([f"{p['V']:.3f} V ({p['I']:.1f} µA)" for p in peaks['oxidation']]) or 'none'
        red_str = ', '.join([f"{p['V']:.3f} V ({p['I']:.1f} µA)" for p in peaks['reduction']]) or 'none'
        label = conditions[ckey]['label']
        print(f"{label} | {rate} mV/s")
        print(f"  Oxidation peaks : {ox_str}")
        print(f"  Reduction peaks : {red_str}")
        print()

## 5. Annotated CV Plots with Peak Markers

In [ ]:
for ckey, cinfo in conditions.items():
    cmap = cm.get_cmap(cinfo['color_base'])
    rates_available = sorted(all_data.get(ckey, {}).keys())
    if not rates_available:
        continue

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()

    for ax, rate in zip(axes, rates_available):
        color = cmap(0.65)
        scans = all_data[ckey][rate]
        df = scans[-1]
        peaks, I_smooth = find_redox_peaks(df)

        ax.plot(df['V'], df['uA'], color='lightgray', lw=1, label='raw', zorder=1)
        ax.plot(df['V'], I_smooth, color=color, lw=2, label='smoothed', zorder=2)

        for p in peaks['oxidation']:
            ax.scatter(p['V'], p['I'], marker='^', s=100, color='red',
                       zorder=5, label=f"Ox {p['V']:.3f} V")
            ax.annotate(f"Eox={p['V']:.3f}V\n{p['I']:.1f}µA",
                        xy=(p['V'], p['I']), xytext=(8, 8),
                        textcoords='offset points', fontsize=8, color='red')

        for p in peaks['reduction']:
            ax.scatter(p['V'], p['I'], marker='v', s=100, color='blue',
                       zorder=5, label=f"Red {p['V']:.3f} V")
            ax.annotate(f"Ered={p['V']:.3f}V\n{p['I']:.1f}µA",
                        xy=(p['V'], p['I']), xytext=(8, -18),
                        textcoords='offset points', fontsize=8, color='blue')

        # Draw E1/2 midpoints
        for ox in peaks['oxidation']:
            for red in peaks['reduction']:
                e_half = (ox['V'] + red['V']) / 2
                ax.axvline(e_half, color='purple', ls=':', lw=1.2,
                           label=f'E½={e_half:.3f}V')

        ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
        ax.set_xlabel('Potential (V)')
        ax.set_ylabel('Current (µA)')
        ax.set_title(f'{rate} mV/s')
        ax.legend(loc='best', fontsize=7)
        ax.grid(True, alpha=0.2)

    plt.suptitle(f'{cinfo["label"]}\nPeak-annotated CV (1 M NaCl)', fontsize=13)
    plt.tight_layout()
    fname = f'cv_peaks_{ckey}.png'
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

## 6. Scan-Rate Dependence Analysis

For diffusion-controlled processes: peak current ∝ √(scan rate)  
For surface-adsorbed processes: peak current ∝ scan rate  
A linear fit to log(Ip) vs log(ν) gives the slope `b`:  
- b ≈ 0.5 → diffusion control  
- b ≈ 1.0 → surface adsorption

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ckey, cinfo in conditions.items():
    ox_rates, ox_ips = [], []
    red_rates, red_ips = [], []
    color = condition_colors[ckey]

    for rate in sorted(peak_results.get(ckey, {}).keys()):
        p = peak_results[ckey][rate]
        if p['oxidation']:
            best_ox = max(p['oxidation'], key=lambda x: x['prominence'])
            ox_rates.append(rate)
            ox_ips.append(abs(best_ox['I']))
        if p['reduction']:
            best_red = max(p['reduction'], key=lambda x: x['prominence'])
            red_rates.append(rate)
            red_ips.append(abs(best_red['I']))

    label = cinfo['label']

    # Ip vs sqrt(v)
    if len(ox_rates) >= 2:
        axes[0].plot(np.sqrt(ox_rates), ox_ips, 'o-', color=color, label=f'{label} (ox)')
    if len(red_rates) >= 2:
        axes[0].plot(np.sqrt(red_rates), red_ips, 's--', color=color, label=f'{label} (red)')

    # Log-log slope
    if len(ox_rates) >= 2:
        log_v = np.log(ox_rates)
        log_ip = np.log(ox_ips)
        b, a = np.polyfit(log_v, log_ip, 1)
        axes[1].plot(np.log(ox_rates), np.log(ox_ips), 'o', color=color)
        axes[1].plot(log_v, b * log_v + a, '-', color=color,
                     label=f'{label} (ox) b={b:.2f}')
    if len(red_rates) >= 2:
        log_v = np.log(red_rates)
        log_ip = np.log(red_ips)
        b, a = np.polyfit(log_v, log_ip, 1)
        axes[1].plot(np.log(red_rates), np.log(red_ips), 's', color=color)
        axes[1].plot(log_v, b * log_v + a, '--', color=color,
                     label=f'{label} (red) b={b:.2f}')

axes[0].set_xlabel('√(Scan rate)  [mV/s]^0.5')
axes[0].set_ylabel('|Peak current| (µA)')
axes[0].set_title('Randles–Ševčík: Ip vs √ν')
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.25)

axes[1].set_xlabel('ln(scan rate)')
axes[1].set_ylabel('ln|peak current|')
axes[1].set_title('log-log slope (b)\nb≈0.5 diffusion, b≈1.0 surface')
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.25)

plt.suptitle('Scan-Rate Dependence of Peak Currents', fontsize=13)
plt.tight_layout()
plt.savefig('cv_scan_rate_dependence.png', bbox_inches='tight')
plt.show()

## 7. Summary Table of Redox Potentials

In [ ]:
rows = []
for ckey, cinfo in conditions.items():
    for rate in sorted(peak_results.get(ckey, {}).keys()):
        p = peak_results[ckey][rate]
        ox_peaks = p['oxidation']
        red_peaks = p['reduction']

        best_ox = max(ox_peaks, key=lambda x: x['prominence']) if ox_peaks else None
        best_red = max(red_peaks, key=lambda x: x['prominence']) if red_peaks else None

        e_ox  = f"{best_ox['V']:.3f}"  if best_ox  else '—'
        e_red = f"{best_red['V']:.3f}" if best_red else '—'
        e_half = f"{(best_ox['V'] + best_red['V']) / 2:.3f}" \
                 if (best_ox and best_red) else '—'
        delta_e = f"{abs(best_ox['V'] - best_red['V']):.3f}" \
                  if (best_ox and best_red) else '—'

        rows.append({
            'Condition': cinfo['label'],
            'Scan rate (mV/s)': rate,
            'Eox (V)': e_ox,
            'Ered (V)': e_red,
            'E½ (V)': e_half,
            'ΔE (V)': delta_e,
            'Iox (µA)': f"{best_ox['I']:.1f}"  if best_ox  else '—',
            'Ired (µA)': f"{best_red['I']:.1f}" if best_red else '—',
        })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
summary_df.to_csv('cv_redox_summary.csv', index=False)
print('\nSaved cv_redox_summary.csv')

## 8. Overlay: All Scans per Condition (Stability Check)

In [ ]:
# For each condition at 10 mV/s, overlay all 3 scans to check stability
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (ckey, cinfo) in zip(axes, conditions.items()):
    rate = 10  # slowest rate — most cycles visible
    if rate not in all_data.get(ckey, {}):
        ax.text(0.5, 0.5, 'No data', ha='center', transform=ax.transAxes)
        continue

    scans = all_data[ckey][rate]
    cmap = cm.get_cmap(cinfo['color_base'])
    for i, df in enumerate(scans):
        alpha = 0.5 + 0.5 * i / max(len(scans) - 1, 1)
        ax.plot(df['V'], df['uA'], color=cmap(0.5 + 0.4 * i / max(len(scans) - 1, 1)),
                lw=1.5, alpha=alpha, label=f'Scan {i+1}')

    ax.axhline(0, color='k', lw=0.6, ls='--', alpha=0.4)
    ax.set_xlabel('Potential (V)')
    ax.set_ylabel('Current (µA)')
    ax.set_title(f'{cinfo["label"]}\n10 mV/s — scan stability')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.2)

plt.suptitle('Multi-scan Overlay (10 mV/s) — Electrochemical Stability', fontsize=13)
plt.tight_layout()
plt.savefig('cv_scan_stability.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Publication figure ────────────────────────────────────────────────────────
# 7.08 in = Cell 2-column width; height chosen for a compact, dense layout
fig, axes = plt.subplots(1, 3, figsize=(7.08, 2.7),
                         gridspec_kw={'wspace': 0.50})

for col, (ck, ci) in enumerate(conditions_pub.items()):
    ax      = axes[col]
    palette = ci['colors']                    # light→dark = slow→fast scan rate
    ps      = peak_stats[ck]

    # ── Plot each scan rate ──────────────────────────────────────────────────
    for i, rt in enumerate(scan_rates):
        if (ck, rt) not in records:
            continue
        df = records[(ck, rt)]
        ax.plot(df['V'], df['uA'],
                color=palette[i], lw=0.85, alpha=0.95, zorder=i + 2,
                label=f'{rt}')

    # Zero-current reference
    ax.axhline(0, color='#aaaaaa', lw=0.4, ls='-', zorder=1)

    # ── Oxidation peak band + label ──────────────────────────────────────────
    if len(ps['ox_V']) > 0:
        m, s = float(ps['ox_V'].mean()), float(ps['ox_V'].std())
        ax.axvline(m, color='#c0392b', lw=0.8, ls=':', alpha=0.85, zorder=11)
        ax.axvspan(m - s, m + s, alpha=0.10, color='#c0392b', lw=0, zorder=1)
        # Place text at 96 % of axes height (inside, near top)
        ax.text(m, 0.96,
                f'$E_{{\\rm ox}}$\n${m:+.3f}\\pm{s:.3f}$ V',
                transform=ax.get_xaxis_transform(),
                ha='center', va='top', fontsize=5.5, color='#c0392b',
                linespacing=1.35,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none',
                          alpha=0.75))

    # ── Reduction peak band + label ──────────────────────────────────────────
    if len(ps['red_V']) > 0:
        m, s = float(ps['red_V'].mean()), float(ps['red_V'].std())
        ax.axvline(m, color='#1a5276', lw=0.8, ls=':', alpha=0.85, zorder=11)
        ax.axvspan(m - s, m + s, alpha=0.10, color='#1a5276', lw=0, zorder=1)
        # Place text at 4 % of axes height (inside, near bottom)
        ax.text(m, 0.04,
                f'$E_{{\\rm red}}$\n${m:+.3f}\\pm{s:.3f}$ V',
                transform=ax.get_xaxis_transform(),
                ha='center', va='bottom', fontsize=5.5, color='#1a5276',
                linespacing=1.35,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none',
                          alpha=0.75))

    # ── Scan-rate legend (inside each panel) ─────────────────────────────────
    legend_handles = [
        Line2D([0], [0], color=palette[i], lw=1.3,
               label=f'{rt} mV s\u207b\u00b9')
        for i, rt in enumerate(scan_rates) if (ck, rt) in records
    ]
    leg = ax.legend(handles=legend_handles,
                    loc='upper left', fontsize=5.2,
                    frameon=True, framealpha=0.85,
                    edgecolor='#cccccc', fancybox=False,
                    borderpad=0.5, handlelength=1.4,
                    labelspacing=0.25, handletextpad=0.4)
    leg.get_frame().set_linewidth(0.5)

    # ── Axes labels and title ────────────────────────────────────────────────
    ax.set_xlabel('Potential (V vs. Ag/AgCl)', fontsize=8, labelpad=3)
    if col == 0:
        ax.set_ylabel('Current (µA)', fontsize=8, labelpad=3)

    ax.set_title(f'{ci["label"]}\n'
                 f'\\textit{{{ci["sublabel"]}}}' if False   # keep plain text
                 else f'{ci["label"]}\n({ci["sublabel"]})',
                 fontsize=7.5, pad=5, linespacing=1.35)

    # ── Tick formatting ──────────────────────────────────────────────────────
    ax.tick_params(which='major', labelsize=6.5, pad=2)
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator(2))

    # ── Bold panel label (A / B / C) ─────────────────────────────────────────
    label_x = -0.22 if col == 0 else -0.15
    ax.text(label_x, 1.08, chr(65 + col),
            transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top', ha='left',
            clip_on=False)

# ── Global figure title ───────────────────────────────────────────────────────
fig.text(0.5, 1.03,
         'Cyclic Voltammetry — 3rd Cycle  |  1 M NaCl',
         ha='center', va='bottom', fontsize=8, style='italic')

# ── Save ─────────────────────────────────────────────────────────────────────
for ext in ('pdf', 'png'):
    fig.savefig(f'cv_third_cycle_publication.{ext}',
                bbox_inches='tight', dpi=300)

plt.show()
print('Saved: cv_third_cycle_publication.pdf / .png')

In [ ]:
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
import warnings
warnings.filterwarnings('ignore')

# ── Cell / Nature rcParams ────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'Arial',
    'font.size':          7,
    'axes.labelsize':     8,
    'axes.titlesize':     8,
    'axes.linewidth':     0.75,
    'xtick.labelsize':    7,
    'ytick.labelsize':    7,
    'xtick.major.width':  0.75,
    'ytick.major.width':  0.75,
    'xtick.minor.width':  0.5,
    'ytick.minor.width':  0.5,
    'xtick.major.size':   3,
    'ytick.major.size':   3,
    'xtick.minor.size':   1.5,
    'ytick.minor.size':   1.5,
    'xtick.direction':    'out',
    'ytick.direction':    'out',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'lines.linewidth':    0.9,
    'pdf.fonttype':       42,   # editable text in Illustrator / Inkscape
    'svg.fonttype':       'none',
    'figure.dpi':         150,
})

DATA_DIR  = Path('cyclic_voltametry_exps')
SCAN_IDX  = 2          # 0-based → 3rd cycle
scan_rates = [10, 50, 100, 200]

# ── Condition metadata + per-condition colour palettes ────────────────────────
conditions_pub = {
    'pyrroleDBS': {
        'label':    'PPy/DBS',
        'sublabel': 'bare film',
        'colors':   ['#bdd7e7', '#6baed6', '#2171b5', '#084594'],   # blues: slow→fast
    },
    '1cycle_pyrroleDBS_decellchick': {
        'label':    'PPy/DBS + scaffold',
        'sublabel': '1-cycle coating',
        'colors':   ['#bae4b3', '#74c476', '#238b45', '#005a32'],   # greens
    },
    '0cycle_nopyrroleDBS_decellchick': {
        'label':    'Scaffold only',
        'sublabel': 'control (no PPy)',
        'colors':   ['#fcae91', '#fb6a4a', '#cb181d', '#7f0000'],   # reds
    },
}

# ── CSV parser (UTF-16 BOM, 3-scan layout) ────────────────────────────────────
def parse_cv_file(filepath):
    with open(filepath, 'rb') as f:
        raw = f.read()
    text = (raw.decode('utf-16') if raw[:2] in (b'\xff\xfe', b'\xfe\xff')
            else raw.decode('utf-8', errors='replace'))
    lines = [l.strip() for l in text.splitlines()]
    header_idx = ncols = None
    for i, line in enumerate(lines):
        parts = [p.strip() for p in line.split(',') if p.strip()]
        if parts and all(p in ('V', 'µA', 'mA', 'A', 'uA') for p in parts):
            header_idx, ncols = i, len(parts)
            break
    if header_idx is None:
        raise ValueError(f'No units row in {filepath}')
    rows = []
    for line in lines[header_idx + 1:]:
        nums = []
        for p in line.split(','):
            try:
                nums.append(float(p.strip()))
            except ValueError:
                pass
        if len(nums) >= ncols:
            rows.append(nums[:ncols])
    arr = np.array(rows)
    return [pd.DataFrame({'V': arr[:, s*2], 'uA': arr[:, s*2 + 1]})
            for s in range(ncols // 2)]

# ── Load 3rd cycle from every file ────────────────────────────────────────────
records = {}
for f in sorted(DATA_DIR.glob('*.csv')):
    nm = f.stem
    ck = next((k for k in conditions_pub if k in nm), None)
    rt = next((r for r in scan_rates if f'{r}mVperSec' in nm), None)
    if ck and rt:
        scans = parse_cv_file(f)
        if len(scans) > SCAN_IDX:
            records[(ck, rt)] = scans[SCAN_IDX]

print(f'Loaded {len(records)} third-cycle datasets')
for (ck, rt) in sorted(records.keys(), key=lambda x: (list(conditions_pub).index(x[0]), x[1])):
    print(f'  {conditions_pub[ck]["label"]:30s}  {rt:>3d} mV/s  →  {len(records[(ck,rt)])} pts')

# ── Peak detection ─────────────────────────────────────────────────────────────
def detect_peaks_cv(df, smooth_window=15, smooth_poly=3, prominence_frac=0.05):
    V, I = df['V'].values, df['uA'].values
    wl = min(smooth_window, len(I) - 1)
    if wl % 2 == 0:
        wl -= 1
    wl = max(wl, 5)
    I_s = savgol_filter(I, wl, smooth_poly)
    dV  = np.diff(V)
    fwd = np.where(np.concatenate(([False], dV > 0)))[0]
    rev = np.where(np.concatenate(([False], dV < 0)))[0]
    prom = max(prominence_frac * (I_s.max() - I_s.min()), 5.0)
    ox, red = [], []
    if len(fwd) > 5:
        pks, pp = find_peaks(I_s[fwd], prominence=prom, distance=5)
        ox  = [{'V': V[fwd[p]], 'I': I_s[fwd[p]], 'prom': pp['prominences'][k]}
               for k, p in enumerate(pks)]
    if len(rev) > 5:
        pks, pp = find_peaks(-I_s[rev], prominence=prom, distance=5)
        red = [{'V': V[rev[p]], 'I': I_s[rev[p]], 'prom': pp['prominences'][k]}
               for k, p in enumerate(pks)]
    return ox, red

# ── Aggregate peak statistics per condition (across scan rates) ───────────────
peak_stats = {}
print()
for ck, ci in conditions_pub.items():
    ox_V, red_V = [], []
    for rt in scan_rates:
        if (ck, rt) not in records:
            continue
        ox, red = detect_peaks_cv(records[(ck, rt)])
        if ox:
            ox_V.append(max(ox,  key=lambda x: x['prom'])['V'])
        if red:
            red_V.append(max(red, key=lambda x: x['prom'])['V'])
    peak_stats[ck] = dict(ox_V=np.array(ox_V), red_V=np.array(red_V))
    print(f"{ci['label']}")
    if len(ox_V):
        print(f"  Eox  = {np.mean(ox_V):+.3f} ± {np.std(ox_V):.3f} V  (n={len(ox_V)})")
    if len(red_V):
        print(f"  Ered = {np.mean(red_V):+.3f} ± {np.std(red_V):.3f} V  (n={len(red_V)})")

## 9. Publication Figure — 3rd Cycle, All Conditions & Scan Rates

Cell-style figure showing the stabilized (3rd) CV cycle for every dataset. Redox peak positions (mean ± SD across scan rates) are annotated per condition.